# Retrieval evaluation (không phải production)

Notebook này là **benchmark offline** cho giai đoạn 1: qrels, metrics (Recall / nDCG / MRR), ablation Dense / expansion / rerank / hybrid.

Không import từ package production ngoài pipeline retrieve. Harness (metrics, protocol, runner) nằm hết ở đây.

| File cạnh notebook | Vai trò |
|---|---|
| `queries.json` | 10 query khóa luận + 4 hard pair |
| `qrels.json` | grade 2 / 1 / 0 (đã khớp corpus) |
| `ablation.json` | kết quả lần chạy gần nhất |

Mặc định `RUN_LIVE = False`: chỉ đọc `ablation.json`, không load SPECTER2 / FAISS.

Đặt `RUN_LIVE = True` rồi chạy lại cell ablation khi muốn đo lại trên corpus local.


## 1. Cấu hình


In [1]:
from __future__ import annotations

import json
import logging
import math
from collections.abc import Mapping, Sequence
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Any

import pandas as pd

from research_assistant.config import DEFAULT_ARTIFACTS_DIR, REPO_ROOT, RetrievalConfig

EVAL_DIR = REPO_ROOT / "evaluation"
QUERIES_PATH = EVAL_DIR / "queries.json"
QRELS_PATH = EVAL_DIR / "qrels.json"
ABLATION_PATH = EVAL_DIR / "ablation.json"

# False = chỉ xem kết quả đã lưu. True = chạy lại retrieval (cần corpus + encoder).
RUN_LIVE = True
USE_LLM_EXPANSION = False  # True thì expansion không tái lập được
QUERY_IDS = None  # ví dụ: ["dataset_pruning", "network_pruning"]
VARIANT_NAMES = ["dense_orig", "dense_expanded", "dense_expanded_rerank", "full"]

TOP_K = 25
BROAD_K = 150
POOL = 200
DEVICE = "auto"
KS = (10, 25)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
print("EVAL_DIR:", EVAL_DIR)
print("RUN_LIVE:", RUN_LIVE)


EVAL_DIR: /home/liu/Code/Projects/Research-Assistance-Agent/evaluation
RUN_LIVE: True


## 2. Metrics + protocol

Toàn bộ logic đánh giá (từng nằm ở `research_assistant.evaluation`) — không đưa vào package production.


In [2]:
GradeMap = Mapping[str, int]


def relevant_ids(grades: GradeMap, min_grade: int = 1) -> set[str]:
    return {doc_id for doc_id, grade in grades.items() if grade >= min_grade}


def recall_at_k(ranking: Sequence[str], relevant: set[str], k: int) -> float | None:
    if not relevant:
        return None
    return len(set(ranking[:k]) & relevant) / len(relevant)


def precision_at_k(ranking: Sequence[str], relevant: set[str], k: int) -> float:
    top = ranking[:k]
    if not top:
        return 0.0
    return len(set(top) & relevant) / len(top)


def hit_rate_at_k(ranking: Sequence[str], relevant: set[str], k: int) -> float | None:
    if not relevant:
        return None
    return 1.0 if set(ranking[:k]) & relevant else 0.0


def mrr(ranking: Sequence[str], relevant: set[str]) -> float | None:
    if not relevant:
        return None
    for rank, doc_id in enumerate(ranking, start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0


def _dcg(gains: Sequence[float], k: int) -> float:
    total = 0.0
    for i, gain in enumerate(gains[:k], start=1):
        total += (2.0 ** gain - 1.0) / math.log2(i + 1)
    return total


def ndcg_at_k(ranking: Sequence[str], grades: GradeMap, k: int) -> float | None:
    positive = [g for g in grades.values() if g > 0]
    if not positive:
        return None
    gains = [float(grades.get(doc_id, 0)) for doc_id in ranking[:k]]
    dcg = _dcg(gains, k)
    idcg = _dcg(sorted((float(g) for g in grades.values() if g > 0), reverse=True), k)
    if idcg == 0.0:
        return None
    return dcg / idcg


def judged_precision_at_k(ranking, grades, k, min_grade=1):
    judged = [doc_id for doc_id in ranking[:k] if doc_id in grades]
    if not judged:
        return None
    hits = sum(1 for doc_id in judged if grades[doc_id] >= min_grade)
    return hits / len(judged)


def score_ranking(ranking, grades, ks=KS, min_grade=1):
    relevant = relevant_ids(grades, min_grade=min_grade)
    metrics = {
        "n_relevant": float(len(relevant)),
        "n_judged": float(len(grades)),
        "mrr": mrr(ranking, relevant),
    }
    for k in ks:
        metrics[f"recall@{k}"] = recall_at_k(ranking, relevant, k)
        metrics[f"precision@{k}"] = precision_at_k(ranking, relevant, k)
        metrics[f"ndcg@{k}"] = ndcg_at_k(ranking, grades, k)
        metrics[f"hit_rate@{k}"] = hit_rate_at_k(ranking, relevant, k)
        metrics[f"judged_precision@{k}"] = judged_precision_at_k(ranking, grades, k, min_grade)
    return metrics


def mean_metrics(rows: Sequence[Mapping[str, Any]]) -> dict[str, float]:
    if not rows:
        return {}
    out = {}
    for key in rows[0]:
        values = [
            float(row[key])
            for row in rows
            if isinstance(row.get(key), (int, float)) and not isinstance(row.get(key), bool)
        ]
        if values:
            out[key] = sum(values) / len(values)
    return out


def leak_at_k(ranking, source_grade2, k):
    if not source_grade2:
        return None
    return len(set(ranking[:k]) & source_grade2) / len(source_grade2)


@dataclass(frozen=True)
class EvalQuery:
    id: str
    text: str
    domain: str = ""
    notes: str = ""


@dataclass(frozen=True)
class HardPair:
    a: str
    b: str
    why: str = ""


@dataclass(frozen=True)
class EvalProtocol:
    queries: list[EvalQuery]
    qrels: dict[str, dict[str, int]]
    titles: dict[str, dict[str, str]]
    hard_pairs: list[HardPair]

    def grade2(self, query_id: str) -> set[str]:
        return {doc_id for doc_id, grade in self.qrels.get(query_id, {}).items() if grade >= 2}


def _iter_qrel_rows(rows: Any):
    if isinstance(rows, dict):
        for arxiv_id, grade in rows.items():
            if isinstance(grade, dict):
                yield {"arxiv_id": grade.get("arxiv_id", arxiv_id), **grade}
            else:
                yield {"arxiv_id": arxiv_id, "grade": grade}
        return
    if isinstance(rows, list):
        yield from rows
        return
    raise TypeError(f"Unsupported qrels shape: {type(rows).__name__}")


def load_protocol(queries_path=QUERIES_PATH, qrels_path=QRELS_PATH, query_ids=None) -> EvalProtocol:
    queries_payload = json.loads(Path(queries_path).read_text(encoding="utf-8"))
    qrels_payload = json.loads(Path(qrels_path).read_text(encoding="utf-8"))
    raw_queries = queries_payload["queries"] if isinstance(queries_payload, dict) else queries_payload
    queries = [
        EvalQuery(id=item["id"], text=item["text"], domain=str(item.get("domain") or ""), notes=str(item.get("notes") or ""))
        for item in raw_queries
    ]
    if query_ids:
        wanted = set(query_ids)
        queries = [q for q in queries if q.id in wanted]
        missing = wanted - {q.id for q in queries}
        if missing:
            raise KeyError(f"Unknown query ids: {sorted(missing)}")

    hard_pairs = []
    if isinstance(queries_payload, dict):
        allowed = {q.id for q in queries}
        for item in queries_payload.get("hard_pairs") or []:
            pair = HardPair(a=item["a"], b=item["b"], why=str(item.get("why") or ""))
            if pair.a in allowed and pair.b in allowed:
                hard_pairs.append(pair)

    qrels, titles = {}, {}
    for query_id, rows in qrels_payload.items():
        grades, title_map = {}, {}
        for row in _iter_qrel_rows(rows):
            arxiv_id = str(row["arxiv_id"])
            grades[arxiv_id] = int(row["grade"])
            if row.get("title"):
                title_map[arxiv_id] = str(row["title"])
        qrels[query_id] = grades
        titles[query_id] = title_map
    return EvalProtocol(queries=queries, qrels=qrels, titles=titles, hard_pairs=hard_pairs)


protocol = load_protocol(query_ids=QUERY_IDS)
print(f"{len(protocol.queries)} queries, {sum(len(v) for v in protocol.qrels.values())} qrel rows, {len(protocol.hard_pairs)} hard pairs")
pd.DataFrame([{"id": q.id, "domain": q.domain, "text": q.text} for q in protocol.queries])


10 queries, 78 qrel rows, 4 hard pairs


,id,domain,text
0,dataset_pruning,data-centric,dataset pruning and data subset selection for ...
1,network_pruning,compression,neural network weight pruning and model sparsity
2,data_selection,data-centric,training data selection for deep learning
3,feature_selection,classical-ml,feature selection and variable selection for m...
4,dataset_distillation,data-centric,dataset distillation and dataset condensation
5,knowledge_distillation,compression,knowledge distillation from teacher to student...
6,coreset_selection,data-centric,coreset selection for deep neural network trai...
7,active_learning,data-centric,active learning for deep neural networks
8,data_centric_ai,data-centric,data-centric AI methods for improving training...
9,proxy_maturity,data-centric,proxy maturity in dataset pruning


## 3. Sanity check metrics (thay unit test eval)


In [3]:
ranking = ["a", "x", "b"]
relevant = {"a", "b", "c"}
assert recall_at_k(ranking, relevant, 1) == 1 / 3
assert recall_at_k(ranking, relevant, 3) == 2 / 3
assert mrr(ranking, relevant) == 1.0
assert recall_at_k([], set(), 10) is None

grades = {"a": 2, "c": 1}
dcg = (2**2 - 1) / math.log2(2) + (2**1 - 1) / math.log2(4)
idcg = (2**2 - 1) / math.log2(2) + (2**1 - 1) / math.log2(3)
assert abs(ndcg_at_k(["a", "b", "c"], grades, 3) - dcg / idcg) < 1e-12

assert protocol.qrels["dataset_pruning"]["2205.09329"] == 2
assert protocol.qrels["dataset_pruning"]["2301.00774"] == 0
print("sanity checks passed")


sanity checks passed


## 4. Kết quả ablation đã lưu

Cell này không gọi encoder. Đổi `RUN_LIVE = True` ở trên và chạy section 5 để ghi đè `ablation.json`.


In [4]:
def mean_table(payload: dict) -> pd.DataFrame:
    rows = []
    for name, block in payload["variants"].items():
        mean = block["mean"]
        rows.append(
            {
                "variant": name,
                "label": block["label"],
                "recall@10": mean.get("recall@10"),
                "recall@25": mean.get("recall@25"),
                "ndcg@10": mean.get("ndcg@10"),
                "mrr": mean.get("mrr"),
                "hit@25": mean.get("hit_rate@25"),
            }
        )
    return pd.DataFrame(rows).set_index("variant")


def per_query_recall(payload: dict) -> pd.DataFrame:
    names = list(payload["variants"])
    first = payload["variants"][names[0]]["queries"]
    rows = []
    for i, q in enumerate(first):
        row = {"query": q["query_id"]}
        for name in names:
            row[f"{name}@25"] = payload["variants"][name]["queries"][i]["recall@25"]
        rows.append(row)
    return pd.DataFrame(rows).set_index("query")


def leak_table(payload: dict) -> pd.DataFrame:
    rows = []
    for pair in payload.get("hard_pair_leak@10") or []:
        for variant, leaks in pair["variants"].items():
            row = {"pair": f"{pair['a']} vs {pair['b']}", "variant": variant}
            row.update(leaks)
            rows.append(row)
    return pd.DataFrame(rows)


if ABLATION_PATH.exists():
    payload = json.loads(ABLATION_PATH.read_text(encoding="utf-8"))
    cov = payload["qrel_coverage"]
    print(f"Qrels in corpus: {cov['in_corpus']}/{cov['judgments']}")
    if cov.get("missing"):
        print("Missing:", cov["missing"])
    display(mean_table(payload).style.format("{:.3f}", subset=["recall@10", "recall@25", "ndcg@10", "mrr", "hit@25"]))
    display(per_query_recall(payload).style.format("{:.3f}"))
    leaks = leak_table(payload)
    if not leaks.empty:
        display(leaks)
else:
    print("Chưa có ablation.json — chạy section 5 với RUN_LIVE=True")
    payload = None


Qrels in corpus: 78/78


,label,recall@10,recall@25,ndcg@10,mrr,hit@25
variant,,,,,,
dense_orig,Original query + FAISS,0.121,0.174,0.162,0.285,0.600
dense_expanded,Expanded queries + FAISS,0.135,0.160,0.168,0.300,0.600
dense_expanded_rerank,Expanded + original-query rerank,0.121,0.174,0.162,0.285,0.600
full,Expanded + rerank + keyword RRF,0.121,0.212,0.173,0.367,0.700


,dense_orig@25,dense_expanded@25,dense_expanded_rerank@25,full@25
query,,,,
dataset_pruning,0.375,0.375,0.375,0.375
network_pruning,0.125,0.125,0.125,0.125
data_selection,0.286,0.143,0.286,0.286
feature_selection,0.000,0.000,0.000,0.000
dataset_distillation,0.000,0.000,0.000,0.250
knowledge_distillation,0.000,0.000,0.000,0.000
coreset_selection,0.167,0.167,0.167,0.167
active_learning,0.000,0.000,0.000,0.000
data_centric_ai,0.625,0.625,0.625,0.750


,pair,variant,dataset_pruning_in_network_pruning,network_pruning_in_dataset_pruning,data_selection_in_feature_selection,feature_selection_in_data_selection,dataset_distillation_in_knowledge_distillation,knowledge_distillation_in_dataset_distillation,active_learning_in_coreset_selection,coreset_selection_in_active_learning
0,dataset_pruning vs network_pruning,dense_orig,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,dataset_pruning vs network_pruning,dense_expanded,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,dataset_pruning vs network_pruning,dense_expanded_rerank,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,dataset_pruning vs network_pruning,full,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,data_selection vs feature_selection,dense_orig,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN
5,data_selection vs feature_selection,dense_expanded,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN
6,data_selection vs feature_selection,dense_expanded_rerank,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN
7,data_selection vs feature_selection,full,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN
8,dataset_distillation vs knowledge_distillation,dense_orig,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN
9,dataset_distillation vs knowledge_distillation,dense_expanded,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN


## 5. Chạy ablation trên corpus (tùy chọn)

Dùng `RetrievalPipeline` production. Chỉ bật khi cần đo lại.

Variant:

- `dense_orig` — query gốc + FAISS
- `dense_expanded` — template expansion + FAISS
- `dense_expanded_rerank` — expansion + cosine với topic gốc
- `full` — expansion + rerank + arXiv keyword RRF


In [5]:
@dataclass(frozen=True)
class AblationVariant:
    name: str
    label: str
    n_query_variants: int
    use_rerank: bool
    use_hybrid: bool


VARIANTS = {
    "dense_orig": AblationVariant("dense_orig", "Original query + FAISS", 0, False, False),
    "dense_expanded": AblationVariant("dense_expanded", "Expanded queries + FAISS", 4, False, False),
    "dense_expanded_rerank": AblationVariant("dense_expanded_rerank", "Expanded + original-query rerank", 4, True, False),
    "full": AblationVariant("full", "Expanded + rerank + keyword RRF", 4, True, True),
}


def qrel_coverage(protocol, in_corpus):
    missing, present, total = {}, 0, 0
    selected = {q.id for q in protocol.queries}
    for query_id, grades in protocol.qrels.items():
        if query_id not in selected:
            continue
        dropped = [doc_id for doc_id in grades if doc_id not in in_corpus]
        total += len(grades)
        present += len(grades) - len(dropped)
        if dropped:
            missing[query_id] = dropped
    return {"judgments": total, "in_corpus": present, "missing": missing}


def hard_pair_leaks(protocol, rankings, k=10):
    rows = []
    for pair in protocol.hard_pairs:
        grade2_a, grade2_b = protocol.grade2(pair.a), protocol.grade2(pair.b)
        by_variant = {}
        for variant_name, ranking_by_query in rankings.items():
            rank_a = ranking_by_query.get(pair.a, [])
            rank_b = ranking_by_query.get(pair.b, [])
            by_variant[variant_name] = {
                f"{pair.a}_in_{pair.b}": leak_at_k(rank_b, grade2_a, k),
                f"{pair.b}_in_{pair.a}": leak_at_k(rank_a, grade2_b, k),
            }
        rows.append({"a": pair.a, "b": pair.b, "why": pair.why, "variants": by_variant})
    return rows


def run_ablation(protocol, variant_names=VARIANT_NAMES):
    from research_assistant.retrieval.corpus import ArtifactCorpus
    from research_assistant.retrieval.encoder import Specter2QueryEncoder
    from research_assistant.retrieval.expand import TemplateQueryExpander, build_expander
    from research_assistant.retrieval.pipeline import RetrievalPipeline

    variants = [VARIANTS[name] for name in variant_names]
    base = RetrievalConfig(
        artifacts_dir=DEFAULT_ARTIFACTS_DIR,
        broad_k=BROAD_K,
        candidate_pool_size=POOL,
        top_k=TOP_K,
        device=DEVICE,
        use_citations=False,
    )
    corpus = ArtifactCorpus(base.artifacts_dir, rebuild_index_if_missing=base.rebuild_index_if_missing)
    encoder = Specter2QueryEncoder(device=DEVICE, use_fp16=base.use_fp16)
    expander = build_expander(timeout_s=base.llm_timeout_s) if USE_LLM_EXPANSION else TemplateQueryExpander()
    pipeline = RetrievalPipeline(base, corpus=corpus, encoder=encoder, expander=expander)
    in_corpus = corpus.arxiv_id_to_row()
    coverage = qrel_coverage(protocol, in_corpus)

    per_variant, rankings = {}, {}
    for variant in variants:
        pipeline.config = replace(
            base,
            n_query_variants=variant.n_query_variants,
            use_rerank=variant.use_rerank,
            use_hybrid=variant.use_hybrid,
            use_citations=False,
        )
        query_rows, ranking_by_query = [], {}
        for query in protocol.queries:
            result = pipeline.search(query.text)
            ranking = [paper.arxiv_id for paper in result.papers]
            ranking_by_query[query.id] = ranking
            grades = {doc_id: g for doc_id, g in protocol.qrels.get(query.id, {}).items() if doc_id in in_corpus}
            metrics = score_ranking(ranking, grades, ks=KS)
            metrics["seconds"] = result.metrics.get("seconds")
            query_rows.append({"query_id": query.id, "topic": query.text, **metrics})
            print(variant.name, query.id, f"recall@25={metrics['recall@25']}")
        per_variant[variant.name] = {
            "label": variant.label,
            "config": {
                "n_query_variants": variant.n_query_variants,
                "use_rerank": variant.use_rerank,
                "use_hybrid": variant.use_hybrid,
            },
            "mean": mean_metrics(query_rows),
            "queries": query_rows,
        }
        rankings[variant.name] = ranking_by_query

    return {
        "ks": list(KS),
        "qrel_coverage": coverage,
        "variants": per_variant,
        "hard_pair_leak@10": hard_pair_leaks(protocol, rankings, k=10),
    }


if RUN_LIVE:
    payload = run_ablation(protocol, VARIANT_NAMES)
    ABLATION_PATH.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Wrote", ABLATION_PATH)
    display(mean_table(payload).style.format("{:.3f}", subset=["recall@10", "recall@25", "ndcg@10", "mrr", "hit@25"]))
else:
    print("RUN_LIVE=False — bỏ qua corpus/encoder. Đặt True rồi chạy lại cell này để đo.")


INFO faiss.loader: Loading faiss with AVX512 support.
INFO faiss.loader: Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
INFO faiss.loader: Loading faiss with AVX2 support.
INFO faiss.loader: Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
INFO faiss.loader: Loading faiss.
INFO faiss.loader: Successfully loaded faiss.
/home/liu/Code/Projects/Research-Assistance-Agent/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO research_assistant.retrieval.corpus: Loading FAISS index from /home/liu/Code/Projects/Research-Assistance-Agent/data/specter2_artifacts/index/papers_flatip.faiss
INFO research_assistant.retrieval.encoder: Loading SPECTER2 query encoder on cpu
INFO adapter

dense_orig dataset_pruning recall@25=0.375


INFO research_assistant.retrieval.pipeline: Retrieval metrics: {'n_queries': 1, 'unique_after_union': 150, 'after_filters': 150, 'candidate_pool': 150, 'keyword_hits_in_corpus': 0, 'keyword_added_to_pool': 0, 'citation_requested': 0, 'citation_found': 0, 'top_k_overlap_broad_vs_final': 1.0, 'expander': 'template', 'use_rerank': False, 'use_hybrid': False, 'n_query_variants': 0, 'seconds': 1.04}
INFO research_assistant.retrieval.pipeline: Expanded 1 queries via template


dense_orig network_pruning recall@25=0.125


INFO research_assistant.retrieval.pipeline: Retrieval metrics: {'n_queries': 1, 'unique_after_union': 150, 'after_filters': 150, 'candidate_pool': 150, 'keyword_hits_in_corpus': 0, 'keyword_added_to_pool': 0, 'citation_requested': 0, 'citation_found': 0, 'top_k_overlap_broad_vs_final': 1.0, 'expander': 'template', 'use_rerank': False, 'use_hybrid': False, 'n_query_variants': 0, 'seconds': 1.084}
INFO research_assistant.retrieval.pipeline: Expanded 1 queries via template


...

INFO httpx: HTTP Request: GET https://export.arxiv.org/api/query?search_query=all:data-centric+AI+methods+for+improving+training+data+quality&start=0&max_results=50 "HTTP/1.1 200 OK"
INFO research_assistant.retrieval.pipeline: Retrieval metrics: {'n_queries': 5, 'unique_after_union': 416, 'after_filters': 416, 'candidate_pool': 232, 'keyword_hits_in_corpus': 40, 'keyword_added_to_pool': 32, 'citation_requested': 0, 'citation_found': 0, 'top_k_overlap_broad_vs_final': 0.72, 'expander': 'template', 'use_rerank': True, 'use_hybrid': True, 'n_query_variants': 4, 'seconds': 18.364}
INFO research_assistant.retrieval.pipeline: Expanded 5 queries via template


full data_centric_ai recall@25=0.75


INFO httpx: HTTP Request: GET https://export.arxiv.org/api/query?search_query=all:proxy+maturity+in+dataset+pruning&start=0&max_results=50 "HTTP/1.1 200 OK"
INFO research_assistant.retrieval.pipeline: Retrieval metrics: {'n_queries': 5, 'unique_after_union': 420, 'after_filters': 420, 'candidate_pool': 227, 'keyword_hits_in_corpus': 33, 'keyword_added_to_pool': 27, 'citation_requested': 0, 'citation_found': 0, 'top_k_overlap_broad_vs_final': 0.68, 'expander': 'template', 'use_rerank': True, 'use_hybrid': True, 'n_query_variants': 4, 'seconds': 18.132}


full proxy_maturity recall@25=0.16666666666666666
Wrote /home/liu/Code/Projects/Research-Assistance-Agent/evaluation/ablation.json


,label,recall@10,recall@25,ndcg@10,mrr,hit@25
variant,,,,,,
dense_orig,Original query + FAISS,0.121,0.174,0.162,0.285,0.600
dense_expanded,Expanded queries + FAISS,0.135,0.160,0.168,0.300,0.600
dense_expanded_rerank,Expanded + original-query rerank,0.121,0.174,0.162,0.285,0.600
full,Expanded + rerank + keyword RRF,0.121,0.212,0.173,0.367,0.700
